In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
db = pd.read_csv("Bank Customer Churn Prediction.csv")

In [3]:
db.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# data info
db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  object 
 3   gender            10000 non-null  object 
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 937.6+ KB


In [5]:
# check null values
db.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [6]:
# check duplicated values
db.duplicated().sum()

np.int64(0)

In [7]:
# check unique value
db["country"].unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [8]:
# check unique values
db["gender"].unique()

array(['Female', 'Male'], dtype=object)

In [9]:
# data described
db.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [10]:
# drop extra feature
db.drop("customer_id" , axis = 1 , inplace = True)

In [11]:
# assign x and y features
# x = independent feature
# y = dependent feature
x = db.drop("churn" , axis = 1)
y = db["churn"]

In [12]:
cat_cols = x.select_dtypes(include=['object']).columns
num_cols = x.select_dtypes(include=['int64', 'float64']).columns

In [13]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', sparse_output=False)

encoded = encoder.fit_transform(x[cat_cols])

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled = scaler.fit_transform(x[num_cols])

In [15]:
x_final = np.hstack((encoded, scaled))

In [16]:
x_final

array([[ 0.        ,  0.        ,  0.        , ...,  0.64609167,
         0.97024255,  0.02188649],
       [ 0.        ,  1.        ,  0.        , ..., -1.54776799,
         0.97024255,  0.21653375],
       [ 0.        ,  0.        ,  0.        , ...,  0.64609167,
        -1.03067011,  0.2406869 ],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -1.54776799,
         0.97024255, -1.00864308],
       [ 1.        ,  0.        ,  1.        , ...,  0.64609167,
        -1.03067011, -0.12523071],
       [ 0.        ,  0.        ,  0.        , ...,  0.64609167,
        -1.03067011, -1.07636976]])

In [17]:
cat_columns = encoder.get_feature_names_out(cat_cols)

In [18]:
num_columns = num_cols

In [19]:
all_columns = np.concatenate([cat_columns, num_columns])

In [20]:
x_final = pd.DataFrame(x_final, columns=all_columns)

x_final.head()

,country_Germany,country_Spain,gender_Male,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,0.0,0.0,0.0,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886
1,0.0,1.0,0.0,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534
2,0.0,0.0,0.0,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687
3,0.0,0.0,0.0,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918
4,0.0,1.0,0.0,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276


In [21]:
from sklearn.model_selection import train_test_split

In [22]:
# split x and y into train test 
X_train , X_test , y_train , y_test = train_test_split(x_final , y , test_size = 0.3 , random_state = 42)

In [23]:
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import layers, activations

In [24]:
model = Sequential([
        Dense(20 , input_dim = 11 , activation = 'relu'),
        Dense(10 , activation = 'relu'),
        Dense(5  , activation = 'relu'),
        Dense(1  , activation = 'sigmoid')
])

In [25]:
model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [26]:
history = model.fit(X_train , y_train , epochs = 100 , batch_size = 10 , validation_split = 0.20 , verbose = 1)

Epoch 1/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7968 - loss: 0.4816 - val_accuracy: 0.8193 - val_loss: 0.4213
Epoch 2/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8264 - loss: 0.4093 - val_accuracy: 0.8321 - val_loss: 0.3964
Epoch 3/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8298 - loss: 0.3859 - val_accuracy: 0.8286 - val_loss: 0.3827
Epoch 4/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8327 - loss: 0.3712 - val_accuracy: 0.8271 - val_loss: 0.3754
Epoch 5/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8468 - loss: 0.3605 - val_accuracy: 0.8514 - val_loss: 0.3611
Epoch 6/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8568 - loss: 0.3490 - val_accuracy: 0.8571 - val_loss: 0.3513
Epoch 7/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8591 - loss: 0.3420 - val_accuracy: 0.8579 - val_loss: 0.3499
Epoch 8/100
560/560 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8582 - loss: 0.3399 - val_accu

In [27]:
loss , acc = model.evaluate(X_test , y_test , verbose = 0)

In [28]:
print(acc)
print(loss)

0.8576666712760925
0.3524358868598938


In [29]:
y_pred = model.predict(X_test)

print(y_test[:5])
# [1, 0, 1, 0, 1]

print(y_pred[:5])
# [0.87, 0.12, 0.65, 0.31, 0.91]

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step   
6252    0
4684    0
1731    0
4742    0
4521    0
Name: churn, dtype: int64
[[0.01969178]
 [0.00283288]
 [0.0379146 ]
 [0.06346823]
 [0.1827217 ]]


In [30]:
from sklearn.metrics import precision_score , accuracy_score , recall_score
from sklearn.metrics import classification_report

y_pred = (model.predict(X_test) > 0.3).astype(int)

print(f"Accuracy        : {accuracy_score(y_test , y_pred)}")
print(f"Precision       : {precision_score(y_test , y_pred)}")
print(f"Recall          : {recall_score(y_test , y_pred)}")
print(f"Classification  : {classification_report(y_test, y_pred)}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Accuracy        : 0.827
Precision       : 0.5464949928469242
Recall          : 0.6541095890410958
Classification  :               precision    recall  f1-score   support

           0       0.91      0.87      0.89      2416
           1       0.55      0.65      0.60       584

    accuracy                           0.83      3000
   macro avg       0.73      0.76      0.74      3000
weighted avg       0.84      0.83      0.83      3000

